# EV Charging Availability Prediction - LightGBM Model Training

This notebook demonstrates how to train a LightGBM model for predicting EV charging station availability.

## Features:
- Synthetic telemetry data generation
- Feature engineering (occupancy trends, time-based features)
- Model training and evaluation
- Model export for production use

In [ ]:
# Install required packages (run this in a cell if packages are missing)
# !pip install lightgbm pandas numpy scikit-learn joblib matplotlib

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

## Step 1: Generate Synthetic Telemetry Data

In [ ]:
def generate_synthetic_telemetry(num_stations=5, days=30, interval_minutes=5):
    """
    Generate synthetic telemetry data for multiple stations
    
    Simulates realistic patterns:
    - Higher occupancy during commute hours (7-9 AM, 5-7 PM)
    - Lower occupancy at night
    - Weekend vs weekday differences
    - Random variations
    """
    data = []
    start_time = datetime.now() - timedelta(days=days)
    
    for station_id in range(1, num_stations + 1):
        total_connectors = np.random.randint(4, 9)
        current_time = start_time
        
        while current_time < datetime.now():
            hour = current_time.hour
            is_weekend = current_time.weekday() >= 5
            
            # Base occupancy by hour
            if 7 <= hour < 9 or 17 <= hour < 19:
                base_occupancy = 0.7 if not is_weekend else 0.4
            elif 9 <= hour < 17:
                base_occupancy = 0.5
            elif 19 <= hour < 22:
                base_occupancy = 0.4
            else:
                base_occupancy = 0.15
            
            # Add noise
            noise = np.random.uniform(-0.2, 0.2)
            actual_occupancy = np.clip(base_occupancy + noise, 0, 1)
            
            occupied = int(actual_occupancy * total_connectors)
            available = total_connectors - occupied
            
            data.append({
                'station_id': f'station-{station_id}',
                'timestamp': current_time,
                'occupied': occupied,
                'total': total_connectors,
                'available': available,
                'hour': hour,
                'weekday': current_time.weekday(),
                'is_weekend': int(is_weekend)
            })
            
            current_time += timedelta(minutes=interval_minutes)
    
    return pd.DataFrame(data)

# Generate data
df = generate_synthetic_telemetry(num_stations=5, days=30)
print(f"Generated {len(df)} telemetry records")
df.head()

## Step 2: Feature Engineering

In [ ]:
def create_features(df, lookback_periods=[2, 6, 12]):
    """
    Create features for prediction
    
    Features:
    - Moving averages of occupancy
    - Occupancy trends
    - Time-based features
    """
    df = df.sort_values(['station_id', 'timestamp']).copy()
    
    # Calculate rolling averages for each station
    for period in lookback_periods:
        df[f'occupied_ma_{period}'] = df.groupby('station_id')['occupied'].transform(
            lambda x: x.rolling(window=period, min_periods=1).mean()
        )
    
    # Occupancy trend (difference between recent and older)
    df['trend_2_6'] = df['occupied_ma_2'] - df['occupied_ma_6']
    
    # Target: availability in next period (shifted)
    df['target_available'] = df.groupby('station_id')['available'].shift(-1)
    
    # Remove rows without target
    df = df.dropna(subset=['target_available'])
    
    return df

df_features = create_features(df)
print(f"Created features for {len(df_features)} records")
df_features.head()

## Step 3: Prepare Training Data

In [ ]:
# Select features
feature_cols = ['occupied_ma_2', 'occupied_ma_6', 'occupied_ma_12', 'trend_2_6', 'hour', 'weekday']
target_col = 'target_available'

X = df_features[feature_cols]
y = df_features[target_col]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

## Step 4: Train LightGBM Model

In [ ]:
# Create LightGBM datasets
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

# Set parameters
params = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'verbosity': -1
}

# Train model
model = lgb.train(
    params,
    train_data,
    num_boost_round=100,
    valid_sets=[test_data],
    callbacks=[lgb.early_stopping(stopping_rounds=10)]
)

print("\nModel trained successfully!")

## Step 5: Evaluate Model

In [ ]:
# Make predictions
y_pred = model.predict(X_test)

# Calculate metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Mean Absolute Error: {mae:.3f}")
print(f"Root Mean Squared Error: {rmse:.3f}")

# Plot predictions vs actual
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Available Slots')
plt.ylabel('Predicted Available Slots')
plt.title('Model Predictions vs Actual')
plt.grid(True, alpha=0.3)
plt.show()

# Feature importance
importance = model.feature_importance()
feature_importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': importance
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance_df)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df['feature'], feature_importance_df['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Step 6: Save Model for Production

In [ ]:
import os

# Create models directory if it doesn't exist
models_dir = os.path.join('..', 'models')
os.makedirs(models_dir, exist_ok=True)

# Save model
model_path = os.path.join(models_dir, 'lightgbm_model.pkl')
joblib.dump(model, model_path)

print(f"✅ Model saved to: {model_path}")
print("\nThe model can now be used by the FastAPI ML service!")
print("Restart the ML service to load the new model.")

## Step 7: Test Prediction (Optional)

In [ ]:
# Test with sample input
sample_input = pd.DataFrame([{
    'occupied_ma_2': 3.5,
    'occupied_ma_6': 3.2,
    'occupied_ma_12': 3.0,
    'trend_2_6': 0.3,
    'hour': 17,  # 5 PM
    'weekday': 2  # Wednesday
}])

prediction = model.predict(sample_input)[0]
print(f"Predicted available slots: {int(round(prediction))}")